# Chronos-2 on the same 60 frozen cases
Select **Runtime > Change runtime type > T4 GPU**, then run cells in order.
Upload `chronos2_inputs.zip`. The short preflight must pass before the long cell.
Same 24/168-hour targets, up to 2048 hours of history, frozen seasonal baselines.
No fine-tuning, covariates, or cross-learning between origins. Earlier results stay separate.
This notebook does not interpret the results. Download the ZIP and return it for comparison.


In [ ]:
import os, subprocess, sys
from pathlib import Path

ENV = Path("/content/chronos2_forecast_env")
PYTHON = str(ENV / "bin/python")
BOOTSTRAP = Path("/content/chronos2_env_bootstrap")
# Isolated environment, so Colab's own packages cannot conflict.
RUN_ENV = {key: value for key, value in os.environ.items()
           if key not in ("PYTHONPATH", "PYTHONHOME")}
RUN_ENV["PYTHONNOUSERSITE"] = "1"
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade",
                "--target", str(BOOTSTRAP), "virtualenv"], env=RUN_ENV, check=True)
subprocess.run([sys.executable, "-m", "virtualenv", str(ENV)],
               env={**RUN_ENV, "PYTHONPATH": str(BOOTSTRAP)}, check=True)
subprocess.run([PYTHON, "-m", "pip", "install", "--quiet",
                "torch==2.11.0", "chronos-forecasting==2.2.2",
                "transformers==4.57.6", "numpy==2.1.3", "pandas==2.2.3"],
               env=RUN_ENV, check=True)
subprocess.run([PYTHON, "-m", "pip", "check"], env=RUN_ENV, check=True)
print("Setup complete")
with open("/content/chronos2_packages.txt", "w") as stream:
    subprocess.run([PYTHON, "-m", "pip", "freeze"], stdout=stream, env=RUN_ENV, check=True)


In [ ]:
subprocess.run([PYTHON, "-c", "import torch; from chronos import Chronos2Pipeline; "
                "assert torch.cuda.is_available(), 'Select a GPU runtime'; "
                "print(torch.__version__, torch.cuda.get_device_name(0))"], env=RUN_ENV, check=True)


In [ ]:
from google.colab import files
from pathlib import PurePosixPath
import hashlib, zipfile, shutil
ROOT = Path("/content/chronos2_frozen")
ROOT.mkdir(exist_ok=True)
def unpack(payload, destination):
    import io
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        for member in archive.infolist():
            parts = PurePosixPath(member.filename)
            assert not parts.is_absolute() and ".." not in parts.parts and "\\" not in member.filename
            assert (destination / member.filename).resolve().is_relative_to(destination.resolve())
        archive.extractall(destination)
uploaded = files.upload()
assert len(uploaded) == 1, "Choose only chronos2_inputs.zip"
payload = next(iter(uploaded.values()))
assert hashlib.sha256(payload).hexdigest() == "74801c64dd961419711598ff8723aba79dda110be3ed8a75bc46ea0cca983994", "Wrong or changed inputs ZIP"
unpack(payload, ROOT)
del uploaded, payload
RUNNER = str(ROOT / "chronos2_frozen.py")
OUTPUT = ROOT / "results"
OUTPUT.mkdir(exist_ok=True)
shutil.copyfile("/content/chronos2_packages.txt", OUTPUT / "packages.txt")
shutil.copyfile(ROOT / "chronos2_protocol.json", OUTPUT / "chronos2_protocol.json")
for label in ("Q1", "Q2", "Q3", "Q4"):
    subprocess.run([PYTHON, RUNNER, "--root", str(ROOT / label), "--check-only"], env=RUN_ENV, check=True)


In [ ]:
# Small synthetic check only: both horizons, ragged/gapped inputs, batch isolation.
subprocess.run([PYTHON, "-u", RUNNER, "--root", str(ROOT / "Q1"), "--smoke-only"],
               env=RUN_ENV, check=True)


## Optional: restore completed cases after a disconnected runtime
Skip the next cell for a fresh run. To restore, upload a previously downloaded
`chronos2_results.zip`. The runner checks configuration and frozen arrays before reuse.
Only completed cases resume. An interrupted case restarts from its first origin.


In [ ]:
RESTORE = False  # Change to True only when restoring a saved results ZIP.
if RESTORE:
    uploaded = files.upload()
    assert len(uploaded) == 1, "Choose only chronos2_results.zip"
    unpack(next(iter(uploaded.values())), OUTPUT)
    del uploaded


## Long run: 60 cases
Progress shows window, case, origin count, elapsed time and estimated time remaining.
Keep batch size 8 when resuming. If GPU memory fails, use a new output folder and
reduce the batch size consistently for all periods. No automatic configuration changes.
The ZIP download cell also works after an interruption while the runtime remains available.


In [ ]:
BATCH_SIZE = 8
for number, label in enumerate(("Q1", "Q2", "Q3", "Q4"), 1):
    print(f"Window {number}/4: {label}", flush=True)
    subprocess.run([PYTHON, "-u", RUNNER, "--root", str(ROOT / label),
                    "--output", str(OUTPUT / label), "--batch-size", str(BATCH_SIZE)],
                   env=RUN_ENV, check=True)
print("Forecast generation finished. Run the download cell next.")


In [ ]:
import json
for label, expected in (("Q1", 14), ("Q2", 14), ("Q3", 16), ("Q4", 16)):
    marker = OUTPUT / label / "complete.json"
    if marker.exists():
        assert json.loads(marker.read_text())["cases"] == expected
        print(f"{label}: complete ({expected} cases)")
    else:
        print(f"{label}: incomplete; this ZIP is a partial checkpoint")
result_zip = shutil.make_archive("/content/chronos2_results", "zip", OUTPUT)
files.download(result_zip)
